# Lab 12 (Week 3 — Lab 2): Building Models with TensorFlow/Keras

Welcome to Lab 12! In Lab 11, you built a neuron and a forward pass by hand with NumPy. Now you'll use Keras to build, train, evaluate, and save a real neural network.

**For detailed theory on the Sequential API, optimizers, and training arguments, see the Lab 12 PDF and Learning Resources.**

This notebook focuses on the practical, hands-on workflow.

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.

## Getting the Dataset

This lab continues with the Wine dataset from Lab 11, built directly into scikit-learn — no download needed.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

wine = load_wine()
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set:", X_train_scaled.shape)
print("Test set:", X_test_scaled.shape)

## Example 1: Building a Model with the Sequential API

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(13,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])

model.summary()

## Example 2: Compiling & Training

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=8,
    validation_split=0.2,
    verbose=0
)

print("Final training accuracy:", history.history["accuracy"][-1])
print("Final validation accuracy:", history.history["val_accuracy"][-1])

## Example 3: Visualizing Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss over Training")

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy over Training")

plt.tight_layout()
plt.show()

## Example 4: Evaluating and Predicting

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

predictions = model.predict(X_test_scaled, verbose=0)
predicted_classes = predictions.argmax(axis=1)
print("\nFirst 5 predicted classes:", predicted_classes[:5])
print("First 5 actual classes:    ", y_test[:5])

## Example 5: Saving and Loading the Model

In [ ]:
model.save("wine_classifier.keras")
print("Model saved.")

loaded_model = keras.models.load_model("wine_classifier.keras")
loaded_loss, loaded_accuracy = loaded_model.evaluate(X_test_scaled, y_test, verbose=0)
print("Reloaded model test accuracy:", loaded_accuracy)

## Practice Exercise 1: Try a Different Optimizer

**Exercise:** Rebuild the same architecture from Example 1, but compile it with the `"sgd"` optimizer instead of `"adam"`. Train for 30 epochs and compare the test accuracy to Example 2's Adam version.

In [ ]:
model_sgd = None
test_accuracy_sgd = None

### YOUR CODE HERE ###
model_sgd = keras.Sequential([
    layers.Input(shape=(13,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])
model_sgd.compile(optimizer="sgd", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model_sgd.fit(X_train_scaled, y_train, epochs=30, batch_size=8, validation_split=0.2, verbose=0)
_, test_accuracy_sgd = model_sgd.evaluate(X_test_scaled, y_test, verbose=0)
### END ###

print("Test accuracy (SGD):", test_accuracy_sgd)
print("Test accuracy (Adam, Example 4):", test_accuracy)

## Practice Exercise 2: Changing validation_split

**Exercise:** Retrain the original model (Adam optimizer) using `validation_split=0.3` instead of 0.2. Plot the new training/validation loss curves. Does the validation curve look different with less training data?

In [ ]:
### YOUR CODE HERE ###
model_v30 = keras.Sequential([
    layers.Input(shape=(13,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(8, activation="relu"),
    layers.Dense(3, activation="softmax")
])
model_v30.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_v30 = model_v30.fit(X_train_scaled, y_train, epochs=30, batch_size=8,
                            validation_split=0.3, verbose=0)

plt.plot(history_v30.history["loss"], label="Training Loss")
plt.plot(history_v30.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss (validation_split=0.3)")
plt.show()
### END ###

## Practice Exercise 3: Predicting a New Sample After Reloading

**Exercise:** Using the `loaded_model` from Example 5, take one sample from `X_test_scaled` (any index), predict its class, and compare it against the true label in `y_test`.

In [ ]:
sample_index = 0
predicted_class = None

### YOUR CODE HERE ###
sample = X_test_scaled[sample_index:sample_index + 1]
predicted_class = loaded_model.predict(sample, verbose=0).argmax(axis=1)[0]
### END ###

print("Predicted class:", predicted_class)
print("Actual class:", y_test[sample_index])

## Lab Tasks

Complete the following in this notebook, below this cell.

1. **Build & Train Your Own Model**: Design your own architecture (choose the number of layers and neurons), compile it with any optimizer, and train it on the Wine dataset. Report the test accuracy.
2. **Optimizer Comparison**: Train the same architecture with `"adam"`, `"sgd"`, and `"rmsprop"` optimizers (keeping epochs and batch_size constant). Create a small comparison table of test accuracy for each.
3. **Training Curves & Overfitting Check**: Plot the training and validation loss/accuracy curves for your best model. In a markdown cell, state whether you see any signs of overfitting, and how you can tell.
4. **Save, Reload, and Verify**: Save your best model to a file. In a new code cell, reload it and confirm it gives the same test accuracy as before saving — this simulates using a model without retraining it.
5. **Reflection**: In a markdown cell, write 3-4 sentences comparing this Keras workflow to the manual NumPy implementation from Lab 11 — what does Keras handle for you automatically?

In [ ]:
# Task 1
def build_model(optimizer):
    m = keras.Sequential([
        layers.Input(shape=(13,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(3, activation="softmax")
    ])
    m.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

my_model = build_model("adam")
my_history = my_model.fit(X_train_scaled, y_train, epochs=50, batch_size=8,
                          validation_split=0.2, verbose=0)
_, my_acc = my_model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Task 1 test accuracy: {my_acc:.4f}")

In [ ]:
# Task 2
results, models, histories = {}, {}, {}
for opt in ["adam", "sgd", "rmsprop"]:
    models[opt] = build_model(opt)
    histories[opt] = models[opt].fit(X_train_scaled, y_train, epochs=50, batch_size=8,
                                     validation_split=0.2, verbose=0)
    _, results[opt] = models[opt].evaluate(X_test_scaled, y_test, verbose=0)

print("Optimizer | Test Accuracy")
for opt, acc in results.items():
    print(f"{opt:9} | {acc:.4f}")
best_opt = max(results, key=results.get)
print("Best optimizer:", best_opt)

In [ ]:
# Task 3
best_model = models[best_opt]
best_history = histories[best_opt]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(best_history.history["loss"], label="Training Loss")
plt.plot(best_history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title(f"Loss ({best_opt})")
plt.subplot(1, 2, 2)
plt.plot(best_history.history["accuracy"], label="Training Accuracy")
plt.plot(best_history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.title(f"Accuracy ({best_opt})")
plt.tight_layout()
plt.show()

**Task 3 – Overfitting check:** Overfitting would show up as training loss continuing to fall while validation loss flattens or rises, and a widening gap between training and validation accuracy. In my plots the validation loss tracks the training loss closely and the two accuracy curves stay close together, so I see no clear sign of overfitting. (Check this against your own plot: the Wine dataset is small and easy, so a tiny gap is normal.)

In [ ]:
# Task 4
best_model.save("best_wine_model.keras")
_, acc_before = best_model.evaluate(X_test_scaled, y_test, verbose=0)

reloaded = keras.models.load_model("best_wine_model.keras")
_, acc_after = reloaded.evaluate(X_test_scaled, y_test, verbose=0)

print("Accuracy before saving:", acc_before)
print("Accuracy after reloading:", acc_after)
print("Same accuracy:", np.isclose(acc_before, acc_after))

**Task 5 – Reflection:** In Lab 11 I had to write the forward pass, activation functions, and weight handling myself with NumPy, whereas in Keras I describe the architecture in a few lines with `Sequential` and `Dense`. Keras handles weight initialization, backpropagation and gradient computation, and the optimizer updates (Adam, SGD, etc.) automatically inside `model.fit()`. It also tracks loss and accuracy per epoch, splits off validation data, and can save and reload the whole model with one call. This lets me focus on choosing the architecture and hyperparameters instead of the low-level math.